# Adaptive time stepping method

In [ ]:
#    APM41012EP course notebook - Chapter 8 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Adaptive time stepping method
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"
import warnings
warnings.filterwarnings('ignore')

The principle is to adapt the time step to the local dynamics in order to provide an efficient integration strategy.
The user has to provide a tolerance, and the adaptation must rely on an error estimate and must produce a time step such that the local error estimate is lower than the given tolerance. In order to provide such an error estimate, it is proposed to combine two methods of different orders, so that the difference between the two is a conservative error estimate.
In order to minimise the number of function evaluations, one of the methods is fixed and a second method of lower order is built, which uses the same evaluations of the function.

## Method based on the “3/8 rule” method (order 4)

We start from the four-stage Runge-Kutta method of order 4 known as the "$3/8$ rule":

$$
\begin{array}{c|cccc}
0   &         &       &        &    \\
1/3&  1/3  &       &        &    \\
2/3& -1/3  &  1   &        &    \\
1   &   1    &   -1 &  1    &    \\
\hline
    &    1/8 & 3/8  & 3/8  &  1/8  \\
\end{array}
$$

and we build an embedded method of order $3$ with $s+1$ stages, using the values $k_i$, $i=1,2,3,4$ of the previous method:

$$
\widehat y_{1} = y_{0} + \Delta t\,(\widehat b_1k_1 + \ldots, \widehat b_sk_s+ \widehat b_{s+1}\,f(t_1,y_{1})).
$$

The term $\widehat b_{s+1}\,f(t_1,y_{1})$ gives more freedom at no extra cost, since it has to be evaluated for the method of order $4$.

The order conditions are obtained in the classical way, taking into account the term of the additional stage ($a_{s+1,i} = b_i$, $i=1,\ldots,s$), and lead to the four equations:

$$
\begin{array}{rl}
\widehat b_1 + \widehat b_2 +\widehat b_3+ \widehat b_4+\widehat b_5 &= 1\\
\widehat b_2c_2 + \widehat b_3c_3+ \widehat b_4+\widehat b_5 &= 1/2\\
\widehat b_2c_2^2 + \widehat b_3c_3^2+ \widehat b_4+\widehat b_5 &= 1/3\\
\widehat b_3a_{32}c_2 + \widehat b_4(a_{42}c_2+a_{43}c_3) + \widehat b_5/2 &= 1/6.\\
\end{array}
$$

We have five unknowns and four equations. We then choose $\widehat b_5 = 1/6$ in order to obtain:
$$
\widehat b_1=2b_1 - 1/6, \quad \widehat b_2=2(1-c_2)b_2, \quad \widehat b_3=2(1-c_3)b_3, \quad \widehat b_4=0,
$$

We can then write the term $y_1-\widehat y_{1}$ for a time step $\Delta t$:

$$
y_1-\widehat y_{1} = y_1-y(t_0+\Delta t)+ y(t_0+\Delta t)-\widehat y_{1} = O((\Delta t)^{p+1}) + O((\Delta t)^{\widehat p+1}) \approx C\,(\Delta t)^{\widehat p+1}.
$$

The optimal time step $\Delta t_{{\mathrm o\mathrm p\mathrm t}}$ is the one for which:
$$
Tol \approx C\,(\Delta t_{{\mathrm o\mathrm p\mathrm t}})^{\widehat p+1}.
$$

Eliminating the constant $C$ from the last two equations, we obtain:

$$
\Delta t_{{\mathrm o\mathrm p\mathrm t}} = 0.9\, \Delta t\, \sqrt[{\scriptstyle \widehat p+1\atop \vphantom{\bigl(}}\!]{\frac{Tol}{\vert\vert y_1 - \widehat y_{1} \vert\vert}},
$$

where the factor $0.9$ is a safety factor.

In order to improve the robustness of the method, it is preferable to replace the last evaluation of the time step by:

$$
\Delta t_{{\mathrm o\mathrm p\mathrm t}} = \Delta t\, \min\left(5,\max\left(0.2,0.9\sqrt[{\scriptstyle \widehat p+1\atop \vphantom{\bigl(}}\!]{\frac{Tol}{\vert\vert y_1 - \widehat y_{1} \vert\vert}}\right)\right),
$$

and to compute the norm of $y_1-\widehat y_{1}$ as a mix of the relative and absolute $l^2$ norms:

$$
\vert\vert y_1 - \widehat y_{1} \vert\vert = \sqrt{\frac{1}{m}\sum_{j=1}^{m}\left(\frac{y_{j1} - \widehat y_{j1} }{1+max(\vert y_{j0}\vert,\vert  y_{j1}\vert)}\right)^2}
$$

In [ ]:
class embedded_ode_result:

    def __init__(self, y, t, nfev, t_rej, dt, dt_rej, loc_err_est, loc_err):
        self.y = y
        self.t = t
        self.nfev = nfev 
        self.t_rej = t_rej
        self.dt = dt
        self.dt_rej = dt_rej
        self.loc_err_est = loc_err_est
        self.loc_err = loc_err

def rk_embedded(tini, tend, yini, fcn, tol, dt_ini=1.e-2):

    # butcher table of the "3/8 rule" 
    c2 = 1/3; c3 = 2/3; c4 = 1.

    a21 = 1/3; 
    a31 = -1/3; a32 = 1.
    a41 = 1.;   a42 = -1; a43 = 1.

    b1 = 1/8; b2 = 3/8; b3 = 3/8; b4 = 1/8

    # coefficients bhat of the embedded method
    b1hat = 2*b1 - 1/6 
    b2hat = 2*(1-c2)*b2 
    b3hat = 2*(1-c3)*b3
    b4hat = 0.
    b5hat = 1/6
     
    # init all parameters    
    dt = dt_ini

    yini = np.atleast_1d(yini)
    neq = yini.size

    # list
    t_acc = [tini]
    t_rej = []
    dt_acc = []
    dt_rej = []
    y = [yini] 
    loc_err_est = [0.]
    loc_err = [0.]

    tn = tini
    yn = yini

    k5 = np.array(fcn(tn, yn))
    nfev = 1

    while (tn < tend):

        # yn compute with the "3/8 rule" method
        k1 = k5
        k2 = fcn(tn + c2*dt, yn + dt*a21*k1)
        k3 = fcn(tn + c3*dt, yn + dt*(a31*k1 + a32*k2))
        k4 = fcn(tn + c4*dt, yn + dt*(a41*k1 + a42*k2 + a43*k3))

        ynp1 = yn + dt*(b1*k1 + b2*k2 + b3*k3 + b4*k4)

        k5 = np.array(fcn(tn+dt, ynp1))
        nfev += 4

        # yn compute with the embeddded method
        yhatnp1 = yn + dt*(b1hat*k1 + b2hat*k2 + b3hat*k3 + b4hat*k4 + b5hat*k5)

        # compute local error estimation 
        err = np.sqrt(1./neq) * np.linalg.norm((ynp1 - yhatnp1)/(1+np.maximum(np.abs(yn), np.abs(ynp1))))
        dtopt = dt * min(5, max(0.2, 0.9*np.power(tol/err, 1./4.)))

        if (err < tol):

            # compute and save exact local error
            sol = solve_ivp(fcn, (tn, tn+dt), yn, method="DOP853", rtol=1.e-12, atol=1.e-12)
            loc_err.append(np.sqrt(1./neq) * np.linalg.norm(ynp1 - sol.y[:,-1]))
            
            # update for next step
            tn = tn + dt
            yn = ynp1

            # save time, solution and local error estimation
            t_acc.append(tn)
            dt_acc.append(dt) 
            y.append(yn)
            loc_err_est.append(err)

            # compute next dt
            dt = min(dtopt, tend-tn)

        else:
            t_rej.append(tn)
            dt_rej.append(dt) 
            dt = dtopt
            k5 = k1

    y = np.array(np.transpose(np.array(y)), order='F')

    return embedded_ode_result(y, np.array(t_acc), nfev,  np.array(t_rej), np.array(dt_acc), np.array(dt_rej),
                               np.array(loc_err_est), np.array(loc_err))

## Results on the Brusselator model

The dynamics of the oscillating reaction discovered by Belousov and Zhabotinsky can be modelled by the so-called Brusselator model, as a function of two parameters $a$ and $b$:

$$
\left\{\begin{aligned}
d_t y_1 & = a - (b+1) y_1 + y_1^2y_2\\
d_t y_2 & = b y_1 - y_1^2y_2
\end{aligned}\right.
$$

In [ ]:
class brusselator_model:

    def __init__(self, a, b):
        self.a = a
        self.b = b

    def fcn(self, t, y):
        y1, y2 = y
        a = self.a
        b = self.b
        y1_dot = a - (b+1)*y1 + y1*y1*y2
        y2_dot = b*y1 - y1*y1*y2
        return np.array([y1_dot, y2_dot])

    def jac(self, t, y):
        y1, y2 = y
        a = self.a
        b = self.b
        return np.array([[2*y1*y2 -(b+1), y1*y1], 
                         [-2*y1*y2 + b, -y1*y1] ])
    
def plot_sol(tini, tend, yini, tol):

    bm = brusselator_model(a=1, b=4)
    fcn = bm.fcn  

    sol = rk_embedded(tini, tend, yini, fcn, tol)
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1.e-14, atol=1.e-14, t_eval=sol.t)

    glob_err = np.empty(sol.t.size)
    for i in range(sol.t.size):
        glob_err[i] = np.sqrt(1./2) * np.linalg.norm(sol_exa.y[:,i] - sol.y[:,i])


    print("   Number of time step : " + str(sol.t.size-1))
    print("   Number of function evaluations : " + str(sol.nfev))

    fig = make_subplots(rows=3, cols=1, vertical_spacing=0.15)

    color = fig.layout.template.layout.colorway
    marker_y1 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[0]))
    line_y1 = dict(color=color[0])
    marker_y2 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[1]))
    line_y2 = dict(color=color[1])

    marker_3 = dict(size=5, symbol='square', color=color[6])

    fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name='y1', mode='lines+markers', marker=marker_y1, line=line_y1, 
                  legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=sol.y[1], name='y2', mode='lines+markers', marker=marker_y2, line=line_y2, 
                  legendgroup='sol'), row=1, col=1)

    fig.add_trace(go.Scatter(x=sol.t[:-1], y=sol.dt, name='accepted dt', mode='lines+markers', marker_symbol='x', 
                             legendgroup='dt'), row=2, col=1)
    fig.add_trace(go.Scatter(x=sol.t_rej, y=sol.dt_rej, name='rejected dt', mode='markers', marker_symbol='square', 
                             legendgroup='dt'), row=2, col=1)


    fig.add_trace(go.Scatter(x=sol.t, y=sol.loc_err_est, name='est. loc. error', mode='lines+markers', 
                             marker_symbol='x', legendgroup='err'), row=3, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=sol.loc_err, name='loc. error', mode='markers', marker_symbol='square', 
                             legendgroup='err'), row=3, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=glob_err, name='glob. error', mode='markers', marker_symbol='circle', 
                             legendgroup='err'), row=3, col=1)
    fig.add_hline(y=tol, line_width=2, line_dash="dot", row=3, col=1)

    fig.update_layout(legend=dict(tracegroupgap=275), height=1000)
    fig.update_xaxes(title='t', col=1)
    fig.update_yaxes(type='log', exponentformat='e', row=2)
    fig.update_yaxes(type='log', exponentformat='e', row=3, range=[np.log10(tol)-4, np.log10(tol)+2])
    fig.show()

In [ ]:
tini = 0.
tend = 20.

yini = (1.5 , 3)

tol = 1e-2

plot_sol(tini, tend, yini, tol)

## Cost and accuracy

In [ ]:
class ode_result:
    def __init__(self, y, t, nfev):
        self.y = y
        self.t = t
        self.nfev = nfev 


def rk38(tini, tend, nt, yini, fcn):
    
    # butcher table of the "3/8 rule" 
    c2 = 1/3; c3 = 2/3; c4 = 1.

    a21 = 1/3; 
    a31 = -1/3; a32 = 1.
    a41 = 1.;   a42 = -1; a43 = 1.

    b1 = 1/8; b2 = 3/8; b3 = 3/8; b4 = 1/8

    # coefficients bhat of the embedded method
    b1hat = 2*b1 - 1/6 
    b2hat = 2*(1-c2)*b2 
    b3hat = 2*(1-c3)*b3
    b4hat = 0.
    b5hat = 1/6

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt), order='F')
    y[:,0] = yini
    
    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        k1 = fcn(tn, yn)
        k2 = fcn(tn + c2*dt, yn + dt*a21*k1)
        k3 = fcn(tn + c3*dt, yn + dt*(a31*k1 + a32*k2))
        k4 = fcn(tn + c4*dt, yn + dt*(a41*k1 + a42*k2 + a43*k3))
        y[:,it+1] = yn + dt*(b1*k1 + b2*k2 + b3*k3 + b4*k4)

    nfev = 4*(nt-1)

    return ode_result(y, t, nfev)

def plot_cost_precision(tini, tend, nt, yini, tol):
    bm = brusselator_model(a=1, b=4)
    fcn = bm.fcn  

    fe_rk38=[]; norm_rk38=[]

    for nt_i in nt:
        t = np.linspace(tini,tend,nt_i)
        rtol=1e-14; atol=1e-14
        sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=rtol, atol=atol, t_eval=t)
        sol_rk38 = rk38(tini, tend, nt_i, yini, fcn)
        fe_rk38.append(sol_rk38.nfev)
        norm_rk38.append(np.linalg.norm(sol_exa.y-sol_rk38.y) / np.sqrt(nt_i))

    fe_rk_emb=[]; norm_rk_emb=[]

    for tol_i in tol:
        sol_rk_emb = rk_embedded(tini, tend, yini, fcn, tol_i)
        rtol=1e-14; atol=1e-14
        sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=rtol, atol=atol, t_eval=sol_rk_emb.t)
        fe_rk_emb.append(sol_rk_emb.nfev)
        err = 0 
        for i in range(sol_rk_emb.dt.size):
            ldt = sol_rk_emb.dt[i]
            err += ldt * ((sol_exa.y[0,i+1]-sol_rk_emb.y[0,i+1])*(sol_exa.y[0,i+1]-sol_rk_emb.y[0,i+1]) + \
                          (sol_exa.y[1,i+1]-sol_rk_emb.y[1,i+1])*(sol_exa.y[1,i+1]-sol_rk_emb.y[1,i+1])  )
        err = np.sqrt(err)    
        norm_rk_emb.append(err)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fe_rk38, y=norm_rk38, name='3/8 rule', marker_symbol='x'))
    fig.add_trace(go.Scatter(x=fe_rk_emb, y=norm_rk_emb, name='embedded method', marker_symbol='x'))
    fig.update_xaxes(type='log', exponentformat='e', title="Number of function evaluations")
    fig.update_yaxes(type='log', exponentformat='e', title="l2 norm of the error")
    fig.update_layout(legend=dict(x=0.75, bgcolor="rgba(0,0,0,0)"))
    fig.show()

In [ ]:
tini = 0.
tend = 40.

yini = (1.5 , 3)

nt = [1001, 2001, 5001, 10001, 20001, 50001]

tol = [1.e-4, 1.e-6, 1.e-8, 1.e-10, 1e-12, 1e-13]

plot_cost_precision(tini, tend, nt, yini, tol)

### Results on the thermal explosion case with Radau5

We consider a homogeneous chemical reactor described by the evolution of a dimensionless temperature. The variable $\widetilde{\theta}$ is zero at the initial time, it is monotonically increasing and, following a thermal runaway, it will rise abruptly and reach the dimensionless adiabatic flame temperature ${\overline T_r}$ at the end of the reaction. Under some assumptions on the reaction, one arrives at a differential equation of the following form:

$$
d_{\tau} \widetilde{\theta} = f(\widetilde{\theta}) = \exp\Bigg(\frac{\widetilde{\theta}}{1 + \widetilde{\theta} \, / \, \beta}\Bigg) \Bigg(1 - \frac{\widetilde{\theta}}{\overline T_r} \Bigg)
$$

where $\beta$ is directly related to the activation energy ($\beta = \frac{E_a}{RT_0}$), where $E_a$ is the activation energy of the reaction, $R$ the ideal gas constant and $T_0$ the dimensional initial temperature.

In [ ]:
class explosion_with_consumption_1eq_model:

    def __init__(self, Tr, beta):
        self.Tr = Tr
        self.beta = beta 

    def fcn(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        theta_dot = np.exp(theta/(1+(theta/beta))) * (1-theta/Tr) 
        return theta_dot
    
    def jac(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        exp = np.exp(theta/(1+(theta/beta)))
        return (exp/(1+theta/beta)**2) * (1-theta/Tr) - exp/Tr

In [ ]:
def plot_sol_radau(Tr, beta, tini, tend, nt, yini):

    emwc1eq = explosion_with_consumption_1eq_model(Tr, beta)
    fcn = emwc1eq.fcn
    jac = emwc1eq.jac
    
    tol_ref = 1e-9
    sol_ref   = solve_ivp(fcn, (tini, tend), yini, method='Radau', atol=tol_ref, rtol=tol_ref)
    dt_ref = sol_ref.t[1:]-sol_ref.t[:-1]

    tol = 1e-4
    sol_radau = solve_ivp(fcn, (tini, tend), yini, method='Radau', atol=tol, rtol=tol)
    dt_radau = sol_radau.t[1:]-sol_radau.t[:-1]
    
    print(f"Nb time steps for Radau with tol={tol_ref:.2e} (ref sol) : ", sol_ref.t.size - 1)
    print(f"Nb time steps for Radau with tol={tol:.2e}           : ", sol_radau.t.size - 1)
    
    fig =  make_subplots(rows=3, cols=1, vertical_spacing=0.1, 
                         subplot_titles=("Solution", f"dt Radau (tol={tol})", f"dt Radau (tol={tol_ref})"))
    
    fig.add_trace(go.Scatter(x=sol_ref.t, y=sol_ref.y[0], name='ref sol'), row=1, col=1)
    fig.add_trace(go.Scatter(x=sol_radau.t, y=sol_radau.y[0], name='radau'), row=1, col=1)
    
    fig.add_trace(go.Bar(x=sol_radau.t[:-1]+0.5*dt_radau, y=dt_radau, width=dt_radau, name=f'tol={tol}', 
                           showlegend=False, marker_color='rgb(76,114,176)'), row=2, col=1)
    
    fig.add_trace(go.Bar(x=sol_ref.t[:-1]+0.5*dt_ref, y=dt_ref, width=dt_ref, name='tol=1e-9', 
                         showlegend=False, marker_color='rgb(76,114,176)'), row=3, col=1)
                                                                               
    fig.update_yaxes(type="log", exponentformat='e', row=2)
    fig.update_yaxes(type="log", exponentformat='e', row=3)
    fig.update_layout(height=1000, legend=dict(x=0.05, bgcolor="rgba(0,0,0,0)"))
    fig.show()

In [ ]:
Tr = 200.0
beta = 10.

tini = 0.0
tend = 5.0
nt = 100

yini = (0.,)

plot_sol_radau(Tr, beta, tini, tend, nt, yini)